# Phase 1 tournament — report

Reads the artifacts the three tournament rounds left behind. It computes
nothing new: no index is queried, no model is loaded, no GPU is needed.

| round | notebook | what it varied |
| --- | --- | --- |
| 1 | `notebooks/public/06` (`SMOKE=False`) | 8 retrievers — 4 dense embedders, 4 sparse |
| 2 | `notebooks/public/06` (`SMOKE=False`) | reranker: none vs 2 cross-encoders, on the round-1 winners |
| 3 | `notebooks/public/08` | chunk size/overlap: 256/32, 512/64, 1024/128 |

Every run scored the same 281 questions from the 50 seeded development
articles. That is the **tuning set**: every choice below was made by looking
at it. The other 150 articles and their questions are **held out** and have
not been touched — they are the robustness check, run once, later.

So read every number here as a tuning-set estimate. Twenty-three
configurations were compared on these same 281 questions and the best was
kept, which biases the winner's score upward: some of the margin that made it
win is fitted to this particular question sample. The held-out run is what
turns the number below into an honest one.

Both question variants are reported:

- **original** — the question as the NewsQA annotator typed it, anchors
  (`"the"`, `"he"`, …) and all.
- **resolved** — the same question with its anchor resolved. This is the
  deployment-realistic set; a user asking a chatbot writes a standalone
  question. Read the gap between the two as the ceiling and floor of the
  same system, not as two systems.

Cut-offs stop at @5 because these rounds ran with `rerank_top_n=5`. @7 needs
notebook 12, which raised it to 7.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
RESULTS = PROJECT_ROOT / 'reports/phase1'

round1 = pd.read_csv(RESULTS / 'round1.csv')
round2 = pd.read_csv(RESULTS / 'round2.csv')
round3 = pd.read_csv(RESULTS / 'round3.csv')

# Despite the .jsonl suffix these are single pretty-printed objects, not one
# record per line, so json.loads on the whole file is right.
def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding='utf-8'))

round1_winners = load_json('round1_winners.jsonl')
winner_lock = load_json('winner_lock.jsonl')
round3_winner = load_json('round3_resolved_winner.jsonl')

CUTOFFS = (1, 3, 5)
VARIANTS = ('original', 'resolved')
VARIANT_COLOR = {'original': '#c44e52', 'resolved': '#4c72b0'}

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.axisbelow': True, 'figure.autolayout': True})

# Every run must have scored the full question set, or the comparison is void.
n_samples = set(round1['retrieval.n_samples']) | set(round2['retrieval.n_samples']) | set(round3['retrieval.n_samples'])
assert n_samples == {281}, n_samples
assert (pd.concat([round1, round2, round3])['coverage.success_rate'] == 1.0).all()

print(f'{len(round1)} round-1 runs, {len(round2)} round-2 runs, {len(round3)} round-3 runs')
print(f'{n_samples.pop()} questions each, 100% coverage, partition={set(round1["partition"]).pop()}')

In [ ]:
def reranker_label(row):
    # 'noop' means the raw retriever ranking survived; the two cross-encoders
    # are only told apart by their model id.
    if row['reranker'] == 'noop':
        return 'none'
    return str(row.get('reranker_model', '')).rsplit('/', 1)[-1]


def metrics(row):
    out = {f'{m}@{k}': row[f'retrieval.{m}@{k}'] for m in ('hit_rate', 'mrr', 'ndcg') for k in CUTOFFS}
    out['p50_ms'] = row['latency.total.p50_ms']
    return out


def one(frame, **where):
    """Exactly one row matching the filters, or an assertion rather than a silent mean."""
    mask = pd.Series(True, index=frame.index)
    for key, value in where.items():
        mask &= frame[key] == value
    found = frame[mask]
    assert len(found) == 1, f'{where} matched {len(found)} rows'
    return found.iloc[0]


def leaderboard(frame, key, metric='retrieval.mrr@5'):
    wide = frame.pivot_table(index=key, columns='variant', values=metric)
    return wide.sort_values('resolved', ascending=False).round(4)


def grouped_barh(wide, title, xlabel, ax=None):
    ax = wide.plot(kind='barh', ax=ax, figsize=(9, 0.55 * len(wide) + 2),
                   color=[VARIANT_COLOR.get(c, '#888') for c in wide.columns])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    ax.set_xlim(0, 1)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=2, fontsize=8)
    return ax

## Round 1 — which retriever

Eight indexes, no reranker, `top_k=10`. Four dense embedders against four
sparse scorers, on identical chunks.

In [ ]:
r1 = round1.assign(model=round1['index'].str.replace('^(dense|sparse)_', '', regex=True))
board1 = leaderboard(r1, 'model')
board1.style.format('{:.4f}').background_gradient(cmap='Blues', axis=None)

In [ ]:
wide = board1.iloc[::-1]  # barh draws bottom-up, so invert to keep the winner on top
grouped_barh(wide, 'Round 1 — MRR@5 by retriever (no reranker)', 'MRR@5')
plt.show()

# Round 2 promoted one index per family. Check that choice against BOTH
# variants, because the tournament ranked on the original questions only.
for family in ('dense', 'sparse'):
    for variant in VARIANTS:
        top = (r1[(r1['retriever'] == family) & (r1['variant'] == variant)]
               .sort_values('retrieval.mrr@5', ascending=False))
        names = ', '.join(f'{r["model"]} {r["retrieval.mrr@5"]:.4f}' for _, r in top.head(2).iterrows())
        print(f'{family:6s} {variant:8s} -> {names}')
print()
print('promoted to round 2:', {k: v['index'] for k, v in round1_winners.items()})

Sparse wins outright, and not narrowly. That is the corpus talking: NewsQA
questions lift proper nouns and numbers straight out of the article, which
is exactly the signal an exact-term scorer is built on and exactly what a
dense embedder averages away.

One caveat on the promotion, visible in the print above: the tournament
ranked on the **original** questions, where `bge-small` edges out `e5-base`
by 0.0018 MRR@5 — noise. On the resolved questions the dense order reverses
and `e5-base` leads. So `dense_best` in rounds 2 and 3 is not necessarily
the strongest dense model for deployment. It does not change any conclusion
here, because every dense variant loses to sparse by a margin ~100x larger
than the gap between them, but it does mean the dense rows below are a
lower bound on dense rather than its ceiling.

In [ ]:
# Accuracy is not free. Plot what each retriever costs per query.
fig, ax = plt.subplots(figsize=(8, 5))
for variant in VARIANTS:
    part = r1[r1['variant'] == variant]
    ax.scatter(part['latency.total.p50_ms'], part['retrieval.mrr@5'],
               s=70, label=variant, color=VARIANT_COLOR[variant],
               marker='o' if variant == 'resolved' else '^')
    for _, row in part.iterrows():
        ax.annotate(row['model'], (row['latency.total.p50_ms'], row['retrieval.mrr@5']),
                    fontsize=7, xytext=(4, 3), textcoords='offset points')
ax.set_xlabel('p50 latency (ms)')
ax.set_ylabel('MRR@5')
ax.set_title('Round 1 — accuracy vs latency')
ax.legend(title='question variant')
plt.show()

## Round 2 — does a reranker pay for itself

The round-1 winners plus a hybrid of the two, each run with no reranker and
with two cross-encoders, `top_k=20` narrowed to 5.

In [ ]:
r2 = round2.assign(reranker=round2.apply(reranker_label, axis=1))
r2_table = (r2.pivot_table(index=['index', 'reranker'], columns='variant',
                           values=['retrieval.mrr@5', 'retrieval.hit_rate@5', 'latency.total.p50_ms'])
              .round(4))
r2_table

In [ ]:
# reranker_delta is already the run's own before/after, so no baseline join.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, variant in zip(axes, VARIANTS):
    part = r2[(r2['variant'] == variant) & (r2['reranker'] != 'none')]
    wide = part.pivot_table(index='index', columns='reranker', values='reranker_delta.delta_mrr@5')
    wide.plot(kind='bar', ax=ax, rot=0, colormap='viridis')
    ax.set_title(f'{variant} questions')
    ax.set_xlabel('')
    ax.bar_label(ax.containers[0], fmt='%.3f', fontsize=7)
    ax.bar_label(ax.containers[1], fmt='%.3f', fontsize=7)
axes[0].set_ylabel('MRR@5 gained by reranking')
fig.suptitle('Round 2 — reranker uplift over the raw retriever')
plt.show()

In [ ]:
# What that uplift costs in milliseconds.
cost = (r2[r2['variant'] == 'resolved']
        .pivot_table(index='index', columns='reranker', values='latency.total.p50_ms')
        .round(1))
gain = (r2[r2['variant'] == 'resolved']
        .pivot_table(index='index', columns='reranker', values='retrieval.mrr@5')
        .round(4))
pd.concat({'p50 ms': cost, 'MRR@5': gain}, axis=1)

Both cross-encoders help every retriever, and `bge-reranker-large` beats
`ms-marco-MiniLM` everywhere — for roughly 3x the latency. Hybrid fusion
does **not** beat sparse alone here: mixing in a weaker dense ranking drags
the fused list down.

## Round 3 — chunk size

The round-2 winner re-run over three chunk geometries. Everything else is
held fixed, so any movement is the chunking.

In [ ]:
r3 = round3.assign(reranker=round3.apply(reranker_label, axis=1),
                   chunk=round3['index'].str.replace('chunk_', '', regex=False))
r3['size'] = r3['index'].str.extract(r'chunk_(\d+)_')[0].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, variant in zip(axes, VARIANTS):
    part = r3[r3['variant'] == variant].sort_values('size')
    for reranker, group in part.groupby('reranker'):
        ax.plot(group['size'], group['retrieval.mrr@5'], marker='o', label=reranker)
        for _, row in group.iterrows():
            ax.annotate(f'{row["retrieval.mrr@5"]:.3f}', (row['size'], row['retrieval.mrr@5']),
                        fontsize=7, xytext=(0, 6), textcoords='offset points', ha='center')
    ax.set_xscale('log', base=2)
    ax.set_xticks(sorted(part['size'].unique()))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel('chunk size (tokens)')
    ax.set_title(f'{variant} questions')
    ax.legend(fontsize=8)
axes[0].set_ylabel('MRR@5')
fig.suptitle('Round 3 — chunk size sweep (sparse + cross-encoder)')
plt.show()

leaderboard(r3[r3['reranker'] == 'bge-reranker-large'], 'chunk')

512/64 wins on both variants. The curve is a shallow inverted U: 256 tokens
splits the evidence span across chunks, 1024 dilutes it with surrounding
text. Since 512/64 was already the default the earlier rounds ran on, round
3 re-elected the incumbent — it confirms the setting rather than improving
on it.

## The three rounds as one funnel

Same retriever family throughout, so the stages actually stack.

In [ ]:
LARGE = 'BAAI/bge-reranker-large'
funnel = []
for variant in VARIANTS:
    for stage, row in [
        ('1 · BGE-M3 sparse', one(round1, index='sparse_bge_m3_sparse', variant=variant)),
        ('2 · + bge-reranker-large', one(round2, index='sparse_best', variant=variant, reranker_model=LARGE)),
        ('3 · + 512/64 chunks', one(round3, index='chunk_512_64', variant=variant, reranker_model=LARGE)),
    ]:
        funnel.append({'stage': stage, 'variant': variant, **metrics(row)})
funnel = pd.DataFrame(funnel)

fig, ax = plt.subplots(figsize=(9, 4.5))
for variant in VARIANTS:
    part = funnel[funnel['variant'] == variant]
    ax.plot(part['stage'], part['mrr@5'], marker='o', color=VARIANT_COLOR[variant], label=variant)
    for _, row in part.iterrows():
        ax.annotate(f'{row["mrr@5"]:.3f}', (row['stage'], row['mrr@5']),
                    fontsize=8, xytext=(0, 8), textcoords='offset points', ha='center')
ax.set_ylabel('MRR@5')
ax.set_ylim(0, 1)
ax.set_title('Phase 1 funnel — what each round bought')
ax.legend(title='question variant')
plt.show()

funnel.set_index(['variant', 'stage']).round(4)

## Original vs resolved

The same system measured against both question sets. The distance from the
diagonal is the anchor penalty, not a retrieval defect.

In [ ]:
gap = r1.pivot_table(index='model', columns='variant', values='retrieval.mrr@5')
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], color='#999', linestyle='--', linewidth=1, label='no penalty')
ax.scatter(gap['original'], gap['resolved'], s=70, color='#4c72b0')
for model, row in gap.iterrows():
    ax.annotate(model, (row['original'], row['resolved']), fontsize=7,
                xytext=(5, -2), textcoords='offset points')
ax.set_xlabel('MRR@5 — original questions')
ax.set_ylabel('MRR@5 — resolved questions')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.set_title('Anchored questions cost every retriever the same way')
ax.legend()
plt.show()

print(f'mean MRR@5 lift from resolving: {(gap["resolved"] - gap["original"]).mean():.4f}')
print(f'range: {(gap["resolved"] - gap["original"]).min():.4f} to {(gap["resolved"] - gap["original"]).max():.4f}')

## Recommendation — the single configuration to lock

Goal stated: maximise retrieval accuracy. Every number below is the
`resolved` variant, which is what a deployed chatbot actually sees, measured
on the tuning set.

In [ ]:
LOCK = {
    'retriever': 'sparse',
    'embedding model': 'BAAI/bge-m3 (sparse/lexical vectors)',
    'reranker': LARGE,
    'chunking': 'recursive, 512 tokens / 64 overlap',
    'top_k': 20,
    'rerank_top_n': 5,
}
winner = one(round3, index='chunk_512_64', variant='resolved', reranker_model=LARGE)

runner_up = {
    'cheapest reranker': one(round3, index='chunk_512_64', variant='resolved',
                             reranker_model='cross-encoder/ms-marco-MiniLM-L-6-v2'),
    'no reranker': one(round1, index='sparse_bge_m3_sparse', variant='resolved'),
    'best dense instead': one(round2, index='dense_best', variant='resolved', reranker_model=LARGE),
    'hybrid instead': one(round2, index='hybrid_best', variant='resolved', reranker_model=LARGE),
}

for key, value in LOCK.items():
    print(f'{key:18s} {value}')
print()
rows = {'LOCKED': winner, **runner_up}
pd.DataFrame({name: metrics(row) for name, row in rows.items()}).T.round(4)

**Lock this:**

| knob | value | why |
| --- | --- | --- |
| embedding model | **BGE-M3, sparse vectors** | one model covers it; beat the best dense embedder by ~0.18 MRR@5 before reranking and stayed ahead after |
| reranker | **`BAAI/bge-reranker-large`** | +0.07 MRR@5 over the raw retriever and ahead of `ms-marco-MiniLM` on every index tested |
| chunking | **recursive, 512 / 64** | best of the three sizes swept, on both question variants |
| `top_k` / `rerank_top_n` | **20 → 5** | what rounds 2 and 3 actually ran; the reranker can only reorder what it is handed |

**Do not** pick these, even though they look reasonable:

- *Hybrid fusion.* It lost to plain sparse in round 2. The dense half is
  weak enough on this corpus that fusing it in costs accuracy and adds
  latency.
- *A dense-only index.* The best dense setup lands well short of sparse, and
  the gap does not close after reranking.
- *`ms-marco-MiniLM` as the reranker.* It is ~3x faster and that is the only
  argument for it. If latency ever becomes the binding constraint this is
  the one knob to trade back — but the stated goal is accuracy.

Not tested, so not claimed: `top_k` was never swept on its own. 20 is
inherited from the round-2 configuration, not measured against 30 or 50.
Widening it is the cheapest untested lever left if the recall@5 ceiling
turns out to bind.

**Still open, in order:**

1. *Notebook 12* — hierarchical chunking and @3/@5/@7 against this same
   setup. The only remaining candidate that could displace 512/64 recursive:
   size and strategy are the last free variables, and size is settled.
2. *The held-out run* — the locked configuration, once, over the 150
   articles never used for any decision. Nothing gets selected on that
   result; it either confirms the tuning-set numbers generalise or it does
   not. Run it after notebook 12, so the configuration is final before it is
   spent — the held-out set stops being held out the moment a choice is made
   on it.